In [0]:
%restart_python

In [0]:
import mlflow

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())

active_run = mlflow.active_run()

if active_run is None:
    print("Active run: None")
else:
    print("Active run:", active_run.info.run_id)
    print("Experiment ID:", active_run.info.experiment_id)

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

print("MLflow client created successfully.")

---
Databricks --> Left Side Bar --> Experoments --> Custom Model Training --> Created an experiment named:  finance-timeseries-ml-platform_LSTM-model-selection

In [0]:
# Listing all the experiments in this account

from mlflow.tracking import MlflowClient

client = MlflowClient()

experiments = client.search_experiments()

for experiment in experiments:
    print(
        f"ID: {experiment.experiment_id}\n"
        f"Name: {experiment.name}\n"
        f"Artifact location: {experiment.artifact_location}\n"
        f"{'-' * 60}"
    )

In [0]:
# Retrieving the information of the specific experiment by name

import mlflow

username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()

experiment_name = (
    f"/Users/{username}/"
    "finance-timeseries-ml-platform_LSTM-model-selection"
)

experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    raise RuntimeError(
        f"MLflow experiment not found: {experiment_name}"
    )

print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Artifact location:", experiment.artifact_location)

In [0]:
import mlflow

username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()

experiment_name = (
    f"/Users/{username}/"
    "finance-timeseries-ml-platform_LSTM-model-selection"
)

# Connecting Python to the specific experiment
mlflow.set_experiment(experiment_name)

experiment = mlflow.get_experiment_by_name(experiment_name)

runs = mlflow.search_runs(
    experiment_names=[experiment_name]
)

print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Artifact location:", experiment.artifact_location)
print("Active run:", mlflow.active_run())
print("Number of runs:", len(runs))

In [0]:
# Creating a sample MLflow Run, as a tiny smoke-test run, inside the connected experiment:

import mlflow

with mlflow.start_run(run_name="mlflow_smoke_test") as run:
    mlflow.log_param("example_parameter", 123)
    mlflow.log_metric("example_metric", 0.95)

    print("Run ID:", run.info.run_id)
    print("Experiment ID:", run.info.experiment_id)

# Note: no active run remains when the `with` block finishes.
print("Active run after context:", mlflow.active_run())

runs = mlflow.search_runs(
    experiment_names=[experiment_name]
)

print("Number of runs:", len(runs))
print(runs[[
    "run_id",
    "status",
    "params.example_parameter",
    "metrics.example_metric",
]])

In [0]:
# Deleting the smoke run:

client = MlflowClient()

run_id = mlflow.search_runs(
    experiment_names=[experiment_name]
).iloc[0]["run_id"]

client.delete_run(run_id)

print("Smoke-test run deleted.")

---
PREPARING THE ENVIRONMENT AND THE DATA
---

In [0]:
from pathlib import Path
import site

project_root = Path.cwd().parent
src_path = project_root / "src"

site.addsitedir(str(src_path))

# print("Project root:", project_root)
# print("Source path:", src_path)

In [0]:
import finance_ml
from finance_ml.pipelines.model_dataset_pipeline import (
    prepare_model_dataset,
)
# print("finance_ml:", finance_ml.__file__)

In [0]:
gold_df = spark.table(
    "finance_ml.gold.account_features"
)

print("Gold rows:", gold_df.count())

In [0]:
dataset = prepare_model_dataset(
    gold_df,
    sequence_length=7,
    stride=1,
    scaling_method="mad",
)

In [0]:
print("X_train:", dataset["X_train"].shape)
print("y_train:", dataset["y_train"].shape)

print("X_validation:", dataset["X_validation"].shape)
print("y_validation:", dataset["y_validation"].shape)

print("X_test:", dataset["X_test"].shape)
print("y_test:", dataset["y_test"].shape)

print(
    "Number of features:",
    len(dataset["final_feature_columns"]),
)

print(
    "Number of classes:",
    len(set(dataset["y_train"].ravel())),
)

In [0]:
# --- Freezing the DATASET configuration (same for all models) ---

# Fixed dataset configuration for LSTM model selection:
SEQUENCE_LENGTH = 7
N_FEATURES = len(dataset["final_feature_columns"])
N_CLASSES = 4
BOUNDARY_WIDTH = 2
SEED = 42

print("Sequence length:", SEQUENCE_LENGTH)
print("Number of features:", N_FEATURES)
print("Number of classes:", N_CLASSES)
print("Boundary width:", BOUNDARY_WIDTH)

---
SETTING THE EXPERIMENT 
---

In [0]:
import mlflow

EXPERIMENT_ID = "2355252254662315"

mlflow.set_experiment(
    experiment_id=EXPERIMENT_ID,
)

experiment = mlflow.get_experiment(EXPERIMENT_ID)

print("Experiment ID:", experiment.experiment_id)
print("Experiment name:", experiment.name)
print("Artifact location:", experiment.artifact_location)
print("Active MLflow run:", mlflow.active_run())

---
RUNS
---

---
Configuration A: The baseline model
---

In [0]:
# --- Defining the first MODEL configuration ---

# 1st candidate configuration:
BASELINE_CONFIG = {
    "lstm_units": (64,),
    "dropout_rate": 0.3,
    "learning_rate": 1e-3,
    "epochs": 50,
    "batch_size": 256,
    "patience": 5,
    "evaluate_test": False,
    "seed": SEED,
}

print(BASELINE_CONFIG)

In [0]:
# Creating the MLflow parameter dictionary

baseline_parameters = {
    "sequence_length": SEQUENCE_LENGTH,
    "n_features": N_FEATURES,
    "n_classes": N_CLASSES,
    "boundary_width": BOUNDARY_WIDTH,
    **BASELINE_CONFIG,
    "lstm_units": str(BASELINE_CONFIG["lstm_units"]),
}

print(baseline_parameters)

In [0]:
# Starting the MLflow run for the first model

from finance_ml.models.experiment import run_lstm_experiment
from finance_ml.models.tracking import log_sequence_experiment

baseline_artifact_dir = (
    project_root
    / "artifacts"
    / "lstm_model_selection"
    / "LSTM_baseline"
)

with mlflow.start_run(
    run_name="LSTM_baseline",
) as run:

    mlflow.set_tag(
        "model_type",
        "LSTM",
    )

    mlflow.set_tag(
        "selection_stage",
        "model_selection",
    )

    print("Run ID:", run.info.run_id)

    baseline_result = run_lstm_experiment(
        x_train=dataset["X_train"],
        y_train=dataset["y_train"],
        metadata_train=dataset["train_metadata"],
        x_validation=dataset["X_validation"],
        y_validation=dataset["y_validation"],
        metadata_validation=dataset["validation_metadata"],
        x_test=dataset["X_test"],
        y_test=dataset["y_test"],
        metadata_test=dataset["test_metadata"],
        sequence_length=SEQUENCE_LENGTH,
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        lstm_units=BASELINE_CONFIG["lstm_units"],
        dropout_rate=BASELINE_CONFIG["dropout_rate"],
        learning_rate=BASELINE_CONFIG["learning_rate"],
        boundary_width=BOUNDARY_WIDTH,
        epochs=BASELINE_CONFIG["epochs"],
        batch_size=BASELINE_CONFIG["batch_size"],
        patience=BASELINE_CONFIG["patience"],
        verbose=1,
        evaluate_test=BASELINE_CONFIG["evaluate_test"],
        seed=BASELINE_CONFIG["seed"],
        artifact_dir=baseline_artifact_dir,
    )

    log_sequence_experiment(
        result=baseline_result,
        parameters=baseline_parameters,
    )

    print("MLflow logging completed.")




In [0]:
"""
We distinguish two validation Macro-F1 values for each model such as the baseline model:


1- The metric monitored during Keras training/early stopping, produced by `run_lstm_experiment()` :
    
    baseline_result.history.history["val_macro_f1"]

2- The final post-training evaluation using our actual sequence-boundary trimming and overlapping-prediction aggregation procedure:

    baseline_result.validation_evaluation["macro_f1"]


If the two values are identical, it would mean our current MLflow tracking design is consistent with the actual experiment.
"""

import numpy as np

print("=== Baseline validation evaluation ===")

for key, value in baseline_result.validation_evaluation.items():
    if hasattr(value, "shape"):
        print(f"{key}: shape={value.shape}")
    elif key == "predictions":
        print(f"{key}: {len(value)} records")
    else:
        print(f"{key}: {value}")

print("\n=== Best training-history validation Macro-F1 ===")

history_val_macro_f1 = baseline_result.history.history["val_macro_f1"]

best_epoch = int(np.argmax(history_val_macro_f1) + 1)
best_history_macro_f1 = float(history_val_macro_f1[best_epoch - 1])

print("Best epoch:", best_epoch)
print("Best history val_macro_f1:", best_history_macro_f1)

In [0]:
# Inspecting the MLflow run

run = mlflow.get_run(
    "4c4d0f3cfa3e444fb08236c570f61497"
)

print("=== Run ===")
print("Run ID:", run.info.run_id)
print("Status:", run.info.status)

print("\n=== Parameters ===")
for key, value in run.data.params.items():
    print(f"{key}: {value}")

print("\n=== Metrics ===")
for key, value in run.data.metrics.items():
    print(f"{key}: {value}")

print("\n=== Tags ===")
for key, value in run.data.tags.items():
    if key in {
        "model_type",
        "selection_stage",
        "best_epoch",
    }:
        print(f"{key}: {value}")

In [0]:
import pandas as pd

history_df = pd.DataFrame(
    baseline_result.history.history
)

history_df.index = history_df.index + 1
history_df.index.name = "epoch"

display(
    history_df[
        [
            "loss",
            "val_loss",
            "accuracy",
            "val_accuracy",
            "macro_f1",
            "val_macro_f1",
        ]
    ]
)

In [0]:
best_epoch = int(
    history_df["val_macro_f1"].idxmax()
)

print("Best epoch:", best_epoch)
print(
    "Best validation Macro-F1:",
    history_df.loc[best_epoch, "val_macro_f1"],
)

print(
    "\nFinal epoch:",
    history_df.index[-1],
)

print(
    "Final training Macro-F1:",
    history_df.iloc[-1]["macro_f1"],
)

print(
    "Final validation Macro-F1:",
    history_df.iloc[-1]["val_macro_f1"],
)

---
Configuration B: 
stronger regularization
"dropout_rate": 0.5,
---

In [0]:
CONFIG_B = {
    "lstm_units": (64,),
    "dropout_rate": 0.50,
    "learning_rate": 1e-3,
    "epochs": 50,
    "batch_size": 256,
    "patience": 5,
    "evaluate_test": False,
    "seed": SEED,
}

parameters_B = {
    "sequence_length": SEQUENCE_LENGTH,
    "n_features": N_FEATURES,
    "n_classes": N_CLASSES,
    "boundary_width": BOUNDARY_WIDTH,
    **CONFIG_B,
    "lstm_units": str(CONFIG_B["lstm_units"]),
}

print(CONFIG_B)
print(parameters_B)

In [0]:
config_b_artifact_dir = (
    project_root
    / "artifacts"
    / "lstm_model_selection"
    / "LSTM_64_dropout_050"
)

with mlflow.start_run(
    run_name="LSTM_64_dropout_050",
) as run:

    mlflow.set_tag(
        "model_type",
        "LSTM",
    )

    mlflow.set_tag(
        "selection_stage",
        "model_selection",
    )

    print("Run ID:", run.info.run_id)

    result_b = run_lstm_experiment(
        x_train=dataset["X_train"],
        y_train=dataset["y_train"],
        metadata_train=dataset["train_metadata"],
        x_validation=dataset["X_validation"],
        y_validation=dataset["y_validation"],
        metadata_validation=dataset["validation_metadata"],
        x_test=dataset["X_test"],
        y_test=dataset["y_test"],
        metadata_test=dataset["test_metadata"],
        sequence_length=SEQUENCE_LENGTH,
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        lstm_units=CONFIG_B["lstm_units"],
        dropout_rate=CONFIG_B["dropout_rate"],
        learning_rate=CONFIG_B["learning_rate"],
        boundary_width=BOUNDARY_WIDTH,
        epochs=CONFIG_B["epochs"],
        batch_size=CONFIG_B["batch_size"],
        patience=CONFIG_B["patience"],
        verbose=1,
        evaluate_test=CONFIG_B["evaluate_test"],
        seed=CONFIG_B["seed"],
        artifact_dir=config_b_artifact_dir,
    )

    log_sequence_experiment(
        result=result_b,
        parameters=parameters_B,
    )

    print("MLflow logging completed.")

In [0]:
print(
    "Validation accuracy:",
    result_b.validation_evaluation["accuracy"],
)

print(
    "Validation Macro-F1:",
    result_b.validation_evaluation["macro_f1"],
)

history_b = result_b.history.history

best_epoch_b = int(
    np.argmax(history_b["val_macro_f1"]) + 1
)

print("Best epoch:", best_epoch_b)

print(
    "Best validation Macro-F1:",
    history_b["val_macro_f1"][best_epoch_b - 1],
)

---
Configuration C: 
Increased model capacity 
"lstm_units": (128,)
---

In [0]:
CONFIG_C = {
    "lstm_units": (128,),
    "dropout_rate": 0.30,
    "learning_rate": 1e-3,
    "epochs": 50,
    "batch_size": 256,
    "patience": 5,
    "evaluate_test": False,
    "seed": SEED,
}

parameters_C = {
    "sequence_length": SEQUENCE_LENGTH,
    "n_features": N_FEATURES,
    "n_classes": N_CLASSES,
    "boundary_width": BOUNDARY_WIDTH,
    **CONFIG_C,
    "lstm_units": str(CONFIG_C["lstm_units"]),
}

print(CONFIG_C)
print(parameters_C)

In [0]:
config_c_artifact_dir = (
    project_root
    / "artifacts"
    / "lstm_model_selection"
    / "LSTM_128"
)

with mlflow.start_run(
    run_name="LSTM_128",
) as run:

    mlflow.set_tag(
        "model_type",
        "LSTM",
    )

    mlflow.set_tag(
        "selection_stage",
        "model_selection",
    )

    print("Run ID:", run.info.run_id)

    result_c = run_lstm_experiment(
        x_train=dataset["X_train"],
        y_train=dataset["y_train"],
        metadata_train=dataset["train_metadata"],
        x_validation=dataset["X_validation"],
        y_validation=dataset["y_validation"],
        metadata_validation=dataset["validation_metadata"],
        x_test=dataset["X_test"],
        y_test=dataset["y_test"],
        metadata_test=dataset["test_metadata"],
        sequence_length=SEQUENCE_LENGTH,
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        lstm_units=CONFIG_C["lstm_units"],
        dropout_rate=CONFIG_C["dropout_rate"],
        learning_rate=CONFIG_C["learning_rate"],
        boundary_width=BOUNDARY_WIDTH,
        epochs=CONFIG_C["epochs"],
        batch_size=CONFIG_C["batch_size"],
        patience=CONFIG_C["patience"],
        verbose=1,
        evaluate_test=CONFIG_C["evaluate_test"],
        seed=CONFIG_C["seed"],
        artifact_dir=config_c_artifact_dir,
    )

    log_sequence_experiment(
        result=result_c,
        parameters=parameters_C,
    )

    print("MLflow logging completed.")

In [0]:
print(
    "Validation accuracy:",
    result_c.validation_evaluation["accuracy"],
)

print(
    "Validation Macro-F1:",
    result_c.validation_evaluation["macro_f1"],
)

history_c = result_c.history.history

best_epoch_c = int(
    np.argmax(history_c["val_macro_f1"]) + 1
)

print("Best epoch:", best_epoch_c)

print(
    "Best validation Macro-F1:",
    history_c["val_macro_f1"][best_epoch_c - 1],
)

---
Configuration D:
increased depth
"lstm_units": (64, 32),
---

In [0]:
CONFIG_D = {
    "lstm_units": (64, 32),
    "dropout_rate": 0.30,
    "learning_rate": 1e-3,
    "epochs": 50,
    "batch_size": 256,
    "patience": 5,
    "evaluate_test": False,
    "seed": SEED,
}

parameters_D = {
    "sequence_length": SEQUENCE_LENGTH,
    "n_features": N_FEATURES,
    "n_classes": N_CLASSES,
    "boundary_width": BOUNDARY_WIDTH,
    **CONFIG_D,
    "lstm_units": str(CONFIG_D["lstm_units"]),
}

print(CONFIG_D)
print(parameters_D)

In [0]:
config_d_artifact_dir = (
    project_root
    / "artifacts"
    / "lstm_model_selection"
    / "LSTM_64_32"
)

with mlflow.start_run(
    run_name="LSTM_64_32",
) as run:

    mlflow.set_tag(
        "model_type",
        "LSTM",
    )

    mlflow.set_tag(
        "selection_stage",
        "model_selection",
    )

    print("Run ID:", run.info.run_id)

    result_d = run_lstm_experiment(
        x_train=dataset["X_train"],
        y_train=dataset["y_train"],
        metadata_train=dataset["train_metadata"],
        x_validation=dataset["X_validation"],
        y_validation=dataset["y_validation"],
        metadata_validation=dataset["validation_metadata"],
        x_test=dataset["X_test"],
        y_test=dataset["y_test"],
        metadata_test=dataset["test_metadata"],
        sequence_length=SEQUENCE_LENGTH,
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        lstm_units=CONFIG_D["lstm_units"],
        dropout_rate=CONFIG_D["dropout_rate"],
        learning_rate=CONFIG_D["learning_rate"],
        boundary_width=BOUNDARY_WIDTH,
        epochs=CONFIG_D["epochs"],
        batch_size=CONFIG_D["batch_size"],
        patience=CONFIG_D["patience"],
        verbose=1,
        evaluate_test=CONFIG_D["evaluate_test"],
        seed=CONFIG_D["seed"],
        artifact_dir=config_d_artifact_dir,
    )

    log_sequence_experiment(
        result=result_d,
        parameters=parameters_D,
    )

    print("MLflow logging completed.")

In [0]:
print(
    "Validation accuracy:",
    result_d.validation_evaluation["accuracy"],
)

print(
    "Validation Macro-F1:",
    result_d.validation_evaluation["macro_f1"],
)

history_d = result_d.history.history

best_epoch_d = int(
    np.argmax(history_d["val_macro_f1"]) + 1
)

print("Best epoch:", best_epoch_d)

print(
    "Best validation Macro-F1:",
    history_d["val_macro_f1"][best_epoch_d - 1],
)

---
Configuration E: lower learning rate
"learning_rate": 5e-4,
---

In [0]:
CONFIG_E = {
    "lstm_units": (64,),
    "dropout_rate": 0.30,
    "learning_rate": 5e-4,
    "epochs": 50,
    "batch_size": 256,
    "patience": 5,
    "evaluate_test": False,
    "seed": SEED,
}

parameters_E = {
    "sequence_length": SEQUENCE_LENGTH,
    "n_features": N_FEATURES,
    "n_classes": N_CLASSES,
    "boundary_width": BOUNDARY_WIDTH,
    **CONFIG_E,
    "lstm_units": str(CONFIG_E["lstm_units"]),
}

print(CONFIG_E)
print(parameters_E)

In [0]:
config_e_artifact_dir = (
    project_root
    / "artifacts"
    / "lstm_model_selection"
    / "LSTM_64_lr_0005"
)

with mlflow.start_run(
    run_name="LSTM_64_lr_0005",
) as run:

    mlflow.set_tag(
        "model_type",
        "LSTM",
    )

    mlflow.set_tag(
        "selection_stage",
        "model_selection",
    )

    print("Run ID:", run.info.run_id)

    result_e = run_lstm_experiment(
        x_train=dataset["X_train"],
        y_train=dataset["y_train"],
        metadata_train=dataset["train_metadata"],
        x_validation=dataset["X_validation"],
        y_validation=dataset["y_validation"],
        metadata_validation=dataset["validation_metadata"],
        x_test=dataset["X_test"],
        y_test=dataset["y_test"],
        metadata_test=dataset["test_metadata"],
        sequence_length=SEQUENCE_LENGTH,
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        lstm_units=CONFIG_E["lstm_units"],
        dropout_rate=CONFIG_E["dropout_rate"],
        learning_rate=CONFIG_E["learning_rate"],
        boundary_width=BOUNDARY_WIDTH,
        epochs=CONFIG_E["epochs"],
        batch_size=CONFIG_E["batch_size"],
        patience=CONFIG_E["patience"],
        verbose=1,
        evaluate_test=CONFIG_E["evaluate_test"],
        seed=CONFIG_E["seed"],
        artifact_dir=config_e_artifact_dir,
    )

    log_sequence_experiment(
        result=result_e,
        parameters=parameters_E,
    )

    print("MLflow logging completed.")

In [0]:
print(
    "Validation accuracy:",
    result_e.validation_evaluation["accuracy"],
)

print(
    "Validation Macro-F1:",
    result_e.validation_evaluation["macro_f1"],
)

history_e = result_e.history.history

best_epoch_e = int(
    np.argmax(history_e["val_macro_f1"]) + 1
)

print("Best epoch:", best_epoch_e)

print(
    "Best validation Macro-F1:",
    history_e["val_macro_f1"][best_epoch_e - 1],
)

---
A–E MLflow comparison
---

In [0]:
runs = mlflow.search_runs(
    experiment_ids=[EXPERIMENT_ID],
)

comparison = runs[
    [
        "run_id",
        "tags.mlflow.runName",
        "params.lstm_units",
        "params.dropout_rate",
        "params.learning_rate",
        "params.seed",
        "metrics.validation_macro_f1",
        "metrics.validation_accuracy",
        "tags.best_epoch",
    ]
].copy()

comparison = comparison.sort_values(
    "metrics.validation_macro_f1",
    ascending=False,
)

display(comparison)

---
Robustness check of Config A according to the seed change from 42 to 43 and 44
---

In [0]:
CONFIG_A43 = {
    "lstm_units": (64,),
    "dropout_rate": 0.30,
    "learning_rate": 1e-3,
    "epochs": 50,
    "batch_size": 256,
    "patience": 5,
    "evaluate_test": False,
    "seed": 43,
}

parameters_A43 = {
    "sequence_length": SEQUENCE_LENGTH,
    "n_features": N_FEATURES,
    "n_classes": N_CLASSES,
    "boundary_width": BOUNDARY_WIDTH,
    **CONFIG_A43,
    "lstm_units": str(CONFIG_A43["lstm_units"]),
}

config_A43_artifact_dir = (
    project_root / "artifacts" / "lstm_model_selection" / "A43"
)

with mlflow.start_run(run_name="A43") as run:
    mlflow.set_tag("model_type", "LSTM")
    mlflow.set_tag("selection_stage", "model_selection")

    result_A43 = run_lstm_experiment(
        x_train=dataset["X_train"],
        y_train=dataset["y_train"],
        metadata_train=dataset["train_metadata"],
        x_validation=dataset["X_validation"],
        y_validation=dataset["y_validation"],
        metadata_validation=dataset["validation_metadata"],
        x_test=dataset["X_test"],
        y_test=dataset["y_test"],
        metadata_test=dataset["test_metadata"],
        sequence_length=SEQUENCE_LENGTH,
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        lstm_units=CONFIG_A43["lstm_units"],
        dropout_rate=CONFIG_A43["dropout_rate"],
        learning_rate=CONFIG_A43["learning_rate"],
        boundary_width=BOUNDARY_WIDTH,
        epochs=CONFIG_A43["epochs"],
        batch_size=CONFIG_A43["batch_size"],
        patience=CONFIG_A43["patience"],
        verbose=1,
        evaluate_test=False,
        seed=CONFIG_A43["seed"],
        artifact_dir=config_A43_artifact_dir,
    )

    log_sequence_experiment(
        result=result_A43,
        parameters=parameters_A43,
    )

In [0]:
CONFIG_A44 = {
    "lstm_units": (64,),
    "dropout_rate": 0.30,
    "learning_rate": 1e-3,
    "epochs": 50,
    "batch_size": 256,
    "patience": 5,
    "evaluate_test": False,
    "seed": 44,
}

parameters_A44 = {
    "sequence_length": SEQUENCE_LENGTH,
    "n_features": N_FEATURES,
    "n_classes": N_CLASSES,
    "boundary_width": BOUNDARY_WIDTH,
    **CONFIG_A44,
    "lstm_units": str(CONFIG_A44["lstm_units"]),
}

config_A44_artifact_dir = (
    project_root
    / "artifacts"
    / "lstm_model_selection"
    / "A44"
)

with mlflow.start_run(run_name="A44") as run:

    mlflow.set_tag(
        "model_type",
        "LSTM",
    )

    mlflow.set_tag(
        "selection_stage",
        "model_selection",
    )

    print("Run ID:", run.info.run_id)

    result_A44 = run_lstm_experiment(
        x_train=dataset["X_train"],
        y_train=dataset["y_train"],
        metadata_train=dataset["train_metadata"],
        x_validation=dataset["X_validation"],
        y_validation=dataset["y_validation"],
        metadata_validation=dataset["validation_metadata"],
        x_test=dataset["X_test"],
        y_test=dataset["y_test"],
        metadata_test=dataset["test_metadata"],
        sequence_length=SEQUENCE_LENGTH,
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        lstm_units=CONFIG_A44["lstm_units"],
        dropout_rate=CONFIG_A44["dropout_rate"],
        learning_rate=CONFIG_A44["learning_rate"],
        boundary_width=BOUNDARY_WIDTH,
        epochs=CONFIG_A44["epochs"],
        batch_size=CONFIG_A44["batch_size"],
        patience=CONFIG_A44["patience"],
        verbose=1,
        evaluate_test=False,
        seed=CONFIG_A44["seed"],
        artifact_dir=config_A44_artifact_dir,
    )

    log_sequence_experiment(
        result=result_A44,
        parameters=parameters_A44,
    )

    print("MLflow logging completed.")

---
Robustness check of Config B according to the seed change from 42 to 43 and 44
---

In [0]:
CONFIG_B43 = {
    "lstm_units": (64,),
    "dropout_rate": 0.50,
    "learning_rate": 1e-3,
    "epochs": 50,
    "batch_size": 256,
    "patience": 5,
    "evaluate_test": False,
    "seed": 43,
}

parameters_B43 = {
    "sequence_length": SEQUENCE_LENGTH,
    "n_features": N_FEATURES,
    "n_classes": N_CLASSES,
    "boundary_width": BOUNDARY_WIDTH,
    **CONFIG_B43,
    "lstm_units": str(CONFIG_B43["lstm_units"]),
}

config_B43_artifact_dir = (
    project_root / "artifacts" / "lstm_model_selection" / "B43"
)

with mlflow.start_run(run_name="B43") as run:
    mlflow.set_tag("model_type", "LSTM")
    mlflow.set_tag("selection_stage", "model_selection")

    print("Run ID:", run.info.run_id)

    result_B43 = run_lstm_experiment(
        x_train=dataset["X_train"],
        y_train=dataset["y_train"],
        metadata_train=dataset["train_metadata"],
        x_validation=dataset["X_validation"],
        y_validation=dataset["y_validation"],
        metadata_validation=dataset["validation_metadata"],
        x_test=dataset["X_test"],
        y_test=dataset["y_test"],
        metadata_test=dataset["test_metadata"],
        sequence_length=SEQUENCE_LENGTH,
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        lstm_units=CONFIG_B43["lstm_units"],
        dropout_rate=CONFIG_B43["dropout_rate"],
        learning_rate=CONFIG_B43["learning_rate"],
        boundary_width=BOUNDARY_WIDTH,
        epochs=CONFIG_B43["epochs"],
        batch_size=CONFIG_B43["batch_size"],
        patience=CONFIG_B43["patience"],
        verbose=1,
        evaluate_test=False,
        seed=CONFIG_B43["seed"],
        artifact_dir=config_B43_artifact_dir,
    )

    log_sequence_experiment(
        result=result_B43,
        parameters=parameters_B43,
    )

    print(
        "Validation Macro-F1:",
        result_B43.validation_evaluation["macro_f1"],
    )

In [0]:
CONFIG_B44 = {
    "lstm_units": (64,),
    "dropout_rate": 0.50,
    "learning_rate": 1e-3,
    "epochs": 50,
    "batch_size": 256,
    "patience": 5,
    "evaluate_test": False,
    "seed": 44,
}

parameters_B44 = {
    "sequence_length": SEQUENCE_LENGTH,
    "n_features": N_FEATURES,
    "n_classes": N_CLASSES,
    "boundary_width": BOUNDARY_WIDTH,
    **CONFIG_B44,
    "lstm_units": str(CONFIG_B44["lstm_units"]),
}

config_B44_artifact_dir = (
    project_root
    / "artifacts"
    / "lstm_model_selection"
    / "B44"
)

with mlflow.start_run(run_name="B44") as run:

    mlflow.set_tag(
        "model_type",
        "LSTM",
    )

    mlflow.set_tag(
        "selection_stage",
        "model_selection",
    )

    print("Run ID:", run.info.run_id)

    result_B44 = run_lstm_experiment(
        x_train=dataset["X_train"],
        y_train=dataset["y_train"],
        metadata_train=dataset["train_metadata"],
        x_validation=dataset["X_validation"],
        y_validation=dataset["y_validation"],
        metadata_validation=dataset["validation_metadata"],
        x_test=dataset["X_test"],
        y_test=dataset["y_test"],
        metadata_test=dataset["test_metadata"],
        sequence_length=SEQUENCE_LENGTH,
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        lstm_units=CONFIG_B44["lstm_units"],
        dropout_rate=CONFIG_B44["dropout_rate"],
        learning_rate=CONFIG_B44["learning_rate"],
        boundary_width=BOUNDARY_WIDTH,
        epochs=CONFIG_B44["epochs"],
        batch_size=CONFIG_B44["batch_size"],
        patience=CONFIG_B44["patience"],
        verbose=1,
        evaluate_test=False,
        seed=CONFIG_B44["seed"],
        artifact_dir=config_B44_artifact_dir,
    )

    log_sequence_experiment(
        result=result_B44,
        parameters=parameters_B44,
    )

    print("MLflow logging completed.")

    print(
        "Validation accuracy:",
        result_B44.validation_evaluation["accuracy"],
    )

    print(
        "Validation Macro-F1:",
        result_B44.validation_evaluation["macro_f1"],
    )

    history_B44 = result_B44.history.history

    best_epoch_B44 = int(
        np.argmax(history_B44["val_macro_f1"]) + 1
    )

    print("Best epoch:", best_epoch_B44)

    print(
        "Best validation Macro-F1:",
        history_B44["val_macro_f1"][best_epoch_B44 - 1],
    )

---
Final run: Running the winner model again (Configuration A), but this time by enbling the test-set evaluation
---

In [0]:
CONFIG_A_FINAL = {
    "lstm_units": (64,),
    "dropout_rate": 0.30,
    "learning_rate": 1e-3,
    "epochs": 50,
    "batch_size": 256,
    "patience": 5,
    "evaluate_test": True,
    "seed": 42,
}

parameters_A_FINAL = {
    "sequence_length": SEQUENCE_LENGTH,
    "n_features": N_FEATURES,
    "n_classes": N_CLASSES,
    "boundary_width": BOUNDARY_WIDTH,
    **CONFIG_A_FINAL,
    "lstm_units": str(CONFIG_A_FINAL["lstm_units"]),
}

final_artifact_dir = (
    project_root
    / "artifacts"
    / "lstm_final"
)

with mlflow.start_run(run_name="A_final_test") as run:

    mlflow.set_tag(
        "model_type",
        "LSTM",
    )

    mlflow.set_tag(
        "selection_stage",
        "final_evaluation",
    )

    print("Run ID:", run.info.run_id)

    result_A_FINAL = run_lstm_experiment(
        x_train=dataset["X_train"],
        y_train=dataset["y_train"],
        metadata_train=dataset["train_metadata"],
        x_validation=dataset["X_validation"],
        y_validation=dataset["y_validation"],
        metadata_validation=dataset["validation_metadata"],
        x_test=dataset["X_test"],
        y_test=dataset["y_test"],
        metadata_test=dataset["test_metadata"],
        sequence_length=SEQUENCE_LENGTH,
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        lstm_units=CONFIG_A_FINAL["lstm_units"],
        dropout_rate=CONFIG_A_FINAL["dropout_rate"],
        learning_rate=CONFIG_A_FINAL["learning_rate"],
        boundary_width=BOUNDARY_WIDTH,
        epochs=CONFIG_A_FINAL["epochs"],
        batch_size=CONFIG_A_FINAL["batch_size"],
        patience=CONFIG_A_FINAL["patience"],
        verbose=1,
        evaluate_test=True,
        seed=CONFIG_A_FINAL["seed"],
        artifact_dir=final_artifact_dir,
    )

    log_sequence_experiment(
        result=result_A_FINAL,
        parameters=parameters_A_FINAL,
    )

    print("MLflow logging completed.")

    print("\n=== Final validation evaluation ===")
    print(
        "Accuracy:",
        result_A_FINAL.validation_evaluation["accuracy"],
    )
    print(
        "Macro-F1:",
        result_A_FINAL.validation_evaluation["macro_f1"],
    )

    print("\n=== Final test evaluation ===")
    print(
        "Accuracy:",
        result_A_FINAL.test_evaluation["accuracy"],
    )
    print(
        "Macro-F1:",
        result_A_FINAL.test_evaluation["macro_f1"],
    )
    print(
        "Confusion matrix:",
        result_A_FINAL.test_evaluation["confusion_matrix"],
    )
    print(
        "Number of test predictions:",
        result_A_FINAL.test_evaluation["n_predictions"],
    )